# Install Dependencies

In [ ]:
!pip install -q openai

# Load Modules

In [ ]:
import pandas as pd
import json
from openai import OpenAI
from sklearn.model_selection import train_test_split
import re

client = OpenAI(api_key='API_KEY')

# Define Functions

In [ ]:
def format_prompt(data):

    """

    Background: This function helps format data into a list of dicts into the required shape for fine tuning

    Params:
    data (list): list of dicts

    Returns:
    training_data_list (list): a list in the proper format for converting to jsonl

    """

    training_data_list = []

    for x in data:

        updated_data = {
            "messages": [
                {
                    "role": "system",
                    "content": x['system_message']
                },
                {
                    "role": "user",
                    "content": x['user_content']
                }
            ]
        }

        training_data_list.append(updated_data)

    print(training_data_list)

    return training_data_list

In [ ]:
def convert_to_jsonl_and_save(data_list, filename):

    """
    Background:
    This function converts the data_list provided into a jsonl file

    Params:
    data_list (list): a list of a dict ready to convert to jsonl
    filename (str): the name of the filename we want to convert

    """

    with open(filename, 'w') as file:
        for data_dict in data_list:
            json_str = json.dumps(data_dict)  # Convert dictionary to JSON string
            file.write(json_str + '\n')  # Write to file with a newline

    print(f"✅ Data successfully written to {filename}")

In [ ]:
def predict(test, model):
  response = client.chat.completions.create(
      model = model,
      messages= test,
      temperature = 0.2,
      max_tokens= 20
  )
  return response.choices[0].message.content

In [ ]:
def store_predictions(test_df, model, test_data):
  print("fine tuned model id is :", model)
  test_df['Prediction']= None

  for index, row in test_df.iterrows():
    test_message = test_data[index]['messages']
    prediction_result = predict(test_message, model)
    test_df.at[index, 'Prediction'] = prediction_result

  test_df.to_csv("predictions.csv")

# Load Model

In [ ]:
model = 'gpt-4o-2024-08-06'

# Load Data

In [ ]:
data = pd.read_excel('sampled_sentiment_data.xlsx')

# Zero Shot

## Prepare Prompt

In [ ]:
data['system_message'] = 'You are an AI assistant specialized in climate change sentiment analysis. Classify the sentiment of the following Arabic sentence based on its emotional tone regarding climate change. Choose only one sentiment between: Positive, Negative, or Neutral for this Arabic sentence.'
data['user_content'] = 'Sentence:\n' + data['Text'] + '\n Predicted Sentiment: '

data2 = data
data = data[['system_message','user_content']]

In [ ]:
data.to_excel('SA-ZeroShot-gpt.xlsx')

In [ ]:
# Convert to JSON String
data = data.to_json(orient='records')

# parse JSON string and convert it into a list of python dictionaries
datadict = json.loads(data)

## Format Prompt

In [ ]:
# format dict into shape for fine tuning
zero_shot_data = format_prompt(datadict)

In [ ]:
# Save as jsonl
convert_to_jsonl_and_save(zero_shot_data, 'SA-ZeroShot.jsonl')

✅ Data successfully written to SA-ZeroShot.jsonl


## Predict

In [ ]:
response = client.chat.completions.create(
      model = model,
      messages= zero_shot_data[7]['messages'],
      temperature = 0.2,
      max_tokens= 20
  )

response.choices[0].message.content

'Neutral'

In [ ]:
true = pd.read_excel('sampled_sentiment_data.xlsx')
y_true = true['sentiment'].values

In [ ]:
zs = pd.DataFrame()

zs['text'] = data2['Text']
zs['Label'] = y_true
zs = zs.reset_index(drop=True)
zs.head()

,text,Label
0,لقد ضلم ماكرون الشعب المصرى وساهم في قمعه وقتل...,Negative
1,وش ذنبه يعيش هالحياة ؟ 💔 ذي نهاية الحب اللي اش...,Negative
2,@mhm8889 @turkialhussini1 في ناس جاتهم امراض م...,Negative
3,أنا بخلص رياضة من هون وبلاقي جارتنا جايبه صحن ...,Negative
4,بقيت احس ان المرض في مصر آخرته الموت مش العلاج...,Negative


In [ ]:
store_predictions(zs, model, zero_shot_data)

fine tuned model id is : gpt-4o-2024-08-06


In [ ]:
pred = pd.read_csv('SA-ZeroShot-predictions-temp0.2.csv')

In [ ]:
pred['Prediction'].value_counts()

,count
Prediction,
Neutral,455
Negative,45
Positive,1


In [ ]:
pred['Prediction'].isnull().sum()

np.int64(0)

In [ ]:
pred['Prediction'].shape[0]

501

In [ ]:
import numpy as np

preds = []
for answer in pred['Prediction']:
    if (
          "Positive" in answer
          or "positive" in answer
          or "pos" in answer
          or "ايجابي" in answer
          or "ايجابية" in answer
      ):
        preds.append("Positive")
    elif (
          "Negative" in answer
          or "negative" in answer
          or "neg" in answer
          or "سلبي" in answer
          or "سلبية" in answer
      ):
        preds.append("Negative")
    elif (
          "Neutral" in answer
          or "neutral" in answer
          or "neu" in answer
          or "حيادي" in answer
          or "حيادية" in answer
      ):
        preds.append("Neutral")
    else:
      print(answer)
      preds.append('None')

In [ ]:
np.unique(preds)

array(['Negative', 'Neutral', 'Positive'], dtype='<U8')

In [ ]:
from sklearn.metrics import (f1_score,
                             precision_score,
                             recall_score,
                             classification_report,
                             confusion_matrix)

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score

labels = ['Positive', 'Negative', 'Neutral']
mapping = {'Neutral': 0, 'Negative': 1, 'Positive':2}
def map_func(x):
    return mapping.get(x, 1)

y_true = np.vectorize(map_func)(zs.Label)
y_pred = np.vectorize(map_func)(preds)

# Calculate accuracy
accuracy = accuracy_score(y_true=y_true, y_pred=y_pred)
print(f'Accuracy: {accuracy:.3f}')

# Generate accuracy report
unique_labels = set(y_true)  # Get unique labels

f1t = f1_score(y_true=y_true, y_pred=y_pred, average = 'weighted')
print('\nf1_score: ', f1t)

prec = precision_score(y_true=y_true, y_pred=y_pred, average = 'weighted')
print('\precision: ', prec)

recall = recall_score(y_true=y_true, y_pred=y_pred, average = 'weighted')
print('\recall: ', recall)

# Generate classification report
class_report = classification_report(y_true=y_true,
                                     y_pred=y_pred,
                                     digits = 4,
                                     target_names=['Neutral', 'Negative', 'Positive'])
print('\nClassification Report:')
print(class_report)

Accuracy: 0.389

f1_score:  0.2844491534808831
\precision:  0.7090761090761091
ecall:  0.38922155688622756

Classification Report:
              precision    recall  f1-score   support

     Neutral     0.3495    0.9521    0.5113       167
    Negative     0.7778    0.2096    0.3302       167
    Positive     1.0000    0.0060    0.0119       167

    accuracy                         0.3892       501
   macro avg     0.7091    0.3892    0.2844       501
weighted avg     0.7091    0.3892    0.2844       501



# Few Shot

In [ ]:
data = pd.read_excel('sampled_sentiment_data.xlsx')

## Prepare Prompt

In [ ]:
data['system_message'] = '''I want to divide this into system message and user contern
You are an AI assistant specialized in sentiment analysis. Classify the sentiment of the following sentence based on its emotional tone. Choose only one sentiment between: Positive, Negative, or Neutral.

Example 1:
كل فرحة تصنعها لغيرك ستعود لك بشكل اجمل صباح الخير ..

Sentiment: Positive

Example 2:
سئمت رؤيتكِ في كل أغنية أسمعها في كل شعر أقرأه في منتصف قهوتي في كل خطوة أخطيها في كل دمعة سئمت

Sentiment: Negative

Example 3:
اذا لم تستطع أن تترك اثرا جميلا في القلوب فلا تزرع فيها آلما لا ينسى.

Sentiment: Neutral'''
data['user_content'] = 'The sentence you need to classify\nSentence:\n' + data['Text'] +'\nPredicted Sentiment:'

data2 = data
data = data[['system_message','user_content']]

In [ ]:
data.to_excel('SA-FewShot-gpt.xlsx')

In [ ]:
# Convert to JSON String
data = data.to_json(orient='records')

# parse JSON string and convert it into a list of python dictionaries
datadict = json.loads(data)

## Format Prompt

In [ ]:
# format dict into shape for fine tuning
few_shot_data = format_prompt(datadict)

# Save as jsonl
convert_to_jsonl_and_save(few_shot_data, 'SA-FewShot.jsonl')

✅ Data successfully written to SA-FewShot.jsonl


## Predict

In [ ]:
response = client.chat.completions.create(
      model = model,
      messages= few_shot_data[7]['messages'],
      temperature = 0.2,
      max_tokens= 20
  )

response.choices[0].message.content

'Negative'

In [ ]:
y_true = data2['sentiment'].values

In [ ]:
fs = pd.DataFrame()

fs['text'] = data2['Text']
fs['Label'] = y_true
fs = fs.reset_index(drop=True)
fs.head()

,text,Label
0,لقد ضلم ماكرون الشعب المصرى وساهم في قمعه وقتل...,Negative
1,وش ذنبه يعيش هالحياة ؟ 💔 ذي نهاية الحب اللي اش...,Negative
2,@mhm8889 @turkialhussini1 في ناس جاتهم امراض م...,Negative
3,أنا بخلص رياضة من هون وبلاقي جارتنا جايبه صحن ...,Negative
4,بقيت احس ان المرض في مصر آخرته الموت مش العلاج...,Negative


In [ ]:
store_predictions(fs, model, few_shot_data)

fine tuned model id is : gpt-4o-2024-08-06


In [ ]:
pred = pd.read_csv('SA-FewShot-predictions-0.2.csv')

In [ ]:
pred['Prediction'].value_counts()

,count
Prediction,
Negative,211
Neutral,158
Positive,132


In [ ]:
pred['Prediction'].isnull().sum()

np.int64(0)

In [ ]:
import numpy as np

preds = []
for answer in pred['Prediction']:
    if (
          "Positive" in answer
          or "positive" in answer
          or "pos" in answer
          or "ايجابي" in answer
          or "ايجابية" in answer
      ):
        preds.append("Positive")
    elif (
          "Negative" in answer
          or "negative" in answer
          or "neg" in answer
          or "سلبي" in answer
          or "سلبية" in answer
      ):
        preds.append("Negative")
    elif (
          "Neutral" in answer
          or "neutral" in answer
          or "neu" in answer
          or "حيادي" in answer
          or "حيادية" in answer
      ):
        preds.append("Neutral")
    else:
      preds.append('None')

In [ ]:
np.unique(preds)

array(['Negative', 'Neutral', 'Positive'], dtype='<U8')

In [ ]:
from sklearn.metrics import (f1_score,
                             precision_score,
                             recall_score,
                             classification_report,
                             confusion_matrix)

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score

labels = ['Positive', 'Negative', 'Neutral']
mapping = {'Neutral': 0, 'Negative': 1, 'Positive':2}
def map_func(x):
    return mapping.get(x, 1)

y_true = np.vectorize(map_func)(fs.Label)
y_pred = np.vectorize(map_func)(preds)

# Calculate accuracy
accuracy = accuracy_score(y_true=y_true, y_pred=y_pred)
print(f'Accuracy: {accuracy:.3f}')

# Generate accuracy report
unique_labels = set(y_true)  # Get unique labels

f1t = f1_score(y_true=y_true, y_pred=y_pred, average = 'weighted')
print('\nf1_score: ', f1t)

prec = precision_score(y_true=y_true, y_pred=y_pred, average = 'weighted')
print('\precision: ', prec)

recall = recall_score(y_true=y_true, y_pred=y_pred, average = 'weighted')
print('\recall: ', recall)

# Generate classification report
class_report = classification_report(y_true=y_true,
                                     y_pred=y_pred,
                                     digits = 4,
                                     target_names=['Neutral', 'Negative', 'Positive'])
print('\nClassification Report:')
print(class_report)

Accuracy: 0.729

f1_score:  0.7230453068134227
\precision:  0.7319517691765578
ecall:  0.7285429141716567

Classification Report:
              precision    recall  f1-score   support

     Neutral     0.6582    0.6228    0.6400       167
    Negative     0.7346    0.9281    0.8201       167
    Positive     0.8030    0.6347    0.7090       167

    accuracy                         0.7285       501
   macro avg     0.7320    0.7285    0.7230       501
weighted avg     0.7320    0.7285    0.7230       501



# CoT

## Prepare Prompt

In [ ]:
data['system_message'] = '''You are an AI assistant specialized in sentiment analysis. Classify the sentiment of the following sentence based on its emotional tone. Choose only one sentiment between: Positive, Negative, or Neutral.

Step 1: Read the sentence
Carefully read the sentence to fully understand its meaning, context, and tone. Consider both explicit statements and any implied emotional cues.

Step 2: Identify Emotionally Charged Language
- Highlight positive language, such as words indicating satisfaction, happiness, or praise (e.g., great, amazing, love, well-done).
- Highlight negative language, such as words indicating dissatisfaction, frustration, or criticism (e.g., terrible, hate, broken, disappointing).
- Neutral statements neither praise nor criticize.

Step 3: Analyze the Emotional Balance
Consider the overall tone and intent of the sentence, including sarcasm or contrast.

Step 4: Determine Sentiment
- If positive sentiment dominates, classify as Positive.
- If negative sentiment dominates, classify as Negative.
- If there is no clear emotional direction or the content is purely factual, classify as Neutral.

Example 1:
كل فرحة تصنعها لغيرك ستعود لك بشكل اجمل صباح الخير ..

Thoughts:
- Positive words: فرحة, اجمل, الخير
- Negative words: None
- Emotional Balance: tweet uses only positive language and promotes an uplifting message about generosity and the return of happiness.
- Sentiment: positive

Predicted Sentiment: Positive

Example 2:
سئمت رؤيتكِ في كل أغنية أسمعها في كل شعر أقرأه في منتصف قهوتي في كل خطوة أخطيها في كل دمعة سئمت


Thoughts:
- Positive word: None
- Negative words: سئمت, دمعة
- Emotional Balance: tweet uses only negative language which revolves around fatigue, sadness, and being overwhelmed by memories.
- Sentiment: Negative

Predicted Sentiment: Negative

Example 3:
اذا لم تستطع أن تترك اثرا جميلا في القلوب فلا تزرع فيها آلما لا ينسى.

Thoughts:
- Positive words: جميلاً
- Negative words: ألماً
- Emotional Balance: tweet mentions both positive and negative outcomes, the overall tone is cautionary and moralistic, not emotionally expressive.
- Sentiment: Neutral
'''
data['user_content'] = 'The sentence you need to classify\nSentence:\n' + data['Text']

data2 = data
data = data[['system_message','user_content']]

In [ ]:
data.to_excel('SA-CoT-gpt.xlsx')

In [ ]:
# Convert to JSON String
data = data.to_json(orient='records')

# parse JSON string and convert it into a list of python dictionaries
datadict = json.loads(data)

## Format Prompt

In [ ]:
# format dict into shape for fine tuning
CoT_data = format_prompt(datadict)

# Save as jsonl
convert_to_jsonl_and_save(CoT_data, 'SA-CoT.jsonl')

[{'messages': [{'role': 'system', 'content': 'You are an AI assistant specialized in sentiment analysis. Classify the sentiment of the following sentence based on its emotional tone. Choose only one sentiment between: Positive, Negative, or Neutral.\n\nStep 1: Read the sentence\nCarefully read the sentence to fully understand its meaning, context, and tone. Consider both explicit statements and any implied emotional cues.\n\nStep 2: Identify Emotionally Charged Language\n- Highlight positive language, such as words indicating satisfaction, happiness, or praise (e.g., great, amazing, love, well-done).\n- Highlight negative language, such as words indicating dissatisfaction, frustration, or criticism (e.g., terrible, hate, broken, disappointing).\n- Neutral statements neither praise nor criticize.\n\nStep 3: Analyze the Emotional Balance\nConsider the overall tone and intent of the sentence, including sarcasm or contrast.\n\nStep 4: Determine Sentiment\n- If positive sentiment dominates,

## Predict

In [ ]:
response = client.chat.completions.create(
      model = model,
      messages= CoT_data[7]['messages'],
      temperature = 0.2,
      max_tokens= 512
  )

response.choices[0].message.content

'Thoughts:\n- Positive words: None\n- Negative words: صعب, محتاج\n- Emotional Balance: The sentence expresses a need for help and mentions being in a difficult situation, indicating a negative emotional tone.\n- Sentiment: Negative\n\nPredicted Sentiment: Negative'

In [ ]:
def predict(test, model):
  response = client.chat.completions.create(
      model = model,
      messages= test,
      temperature = 0.2,
      max_tokens= 512
  )
  return response.choices[0].message.content

In [ ]:
def store_predictions(test_df, model, test_data):
  print("fine tuned model id is :", model)
  test_df['Prediction']= None

  for index, row in test_df.iterrows():
    test_message = test_data[index]['messages']
    prediction_result = predict(test_message, model)
    test_df.at[index, 'Prediction'] = prediction_result

  test_df.to_csv("SA-CoT-predictions-temp0.2.csv")

In [ ]:
y_true = data2['sentiment'].values

In [ ]:
cot = pd.DataFrame()

cot['text'] = data2['Text']
cot['Label'] = y_true
cot = cot.reset_index(drop=True)
cot.head()

,text,Label
0,لقد ضلم ماكرون الشعب المصرى وساهم في قمعه وقتل...,Negative
1,وش ذنبه يعيش هالحياة ؟ 💔 ذي نهاية الحب اللي اش...,Negative
2,@mhm8889 @turkialhussini1 في ناس جاتهم امراض م...,Negative
3,أنا بخلص رياضة من هون وبلاقي جارتنا جايبه صحن ...,Negative
4,بقيت احس ان المرض في مصر آخرته الموت مش العلاج...,Negative


In [ ]:
store_predictions(cot, model, CoT_data)

fine tuned model id is : gpt-4o-2024-08-06


In [ ]:
pred = pd.read_csv('SA-CoT-predictions-temp0.2.csv')

In [ ]:
pred['Prediction'].value_counts()

,count
Prediction,
"Thoughts:\n- Positive words: ثقله, قوه, كبير, كبير, كبير, 💚\n- Negative words: نقص\n- Emotional Balance: The sentence acknowledges a shortcoming (النقص) but emphasizes the strength and importance of the team and its supporters, using positive language and repetition to highlight enthusiasm and support.\n- Sentiment: Positive\n\nPredicted Sentiment: Positive",1
"Thoughts:\n- Positive words: None\n- Negative words: ضلم, قمعه, قتله, عصابة\n- Emotional Balance: The sentence contains strong negative language, expressing dissatisfaction and criticism towards Macron and his actions, as well as a desire for divine retribution.\n- Sentiment: Negative\n\nPredicted Sentiment: Negative",1
"Thoughts:\n- Positive words: None\n- Negative words: ذنبه, 💔, نهاية, اشغلتونا\n- Emotional Balance: The sentence expresses a negative sentiment through words like ""ذنبه"" (fault), ""نهاية"" (end), and the broken heart emoji ""💔"", indicating disappointment or sadness about the outcome of a situation related to love.\n- Sentiment: Negative\n\nPredicted Sentiment: Negative",1
"Thoughts:\n- Positive words: None\n- Negative words: امراض, هم, سكر, ضغط\n- Emotional Balance: The sentence discusses the stress and health issues (diseases, stress, blood pressure) associated with building a house, indicating a negative emotional tone.\n- Sentiment: Negative\n\nPredicted Sentiment: Negative",1
"Thoughts:\n- Positive words: None\n- Negative words: unfair, 😫, 😭😭\n- Emotional Balance: The sentence expresses frustration and disappointment about the situation, using words like ""unfair"" and emoticons indicating sadness and distress.\n- Sentiment: Negative\n\nPredicted Sentiment: Negative",1
...,...
"Thoughts:\n- Positive words: رضاكم, هدفنا (indicating customer satisfaction is a goal, but this is part of a quoted message and not the user's sentiment)\n- Negative words: ماوصلت شحنتي (indicating dissatisfaction due to the shipment not arriving)\n- Emotional Balance: The sentence expresses dissatisfaction with the service as the shipment has not arrived despite payment. The quoted positive message from the company contrasts with the user's negative experience.\n- Sentiment: Negative\n\nPredicted Sentiment: Negative",1
"Thoughts:\n- Positive words: None\n- Negative words: مسروقة (stolen)\n- Emotional Balance: The sentence questions whether certain lands are stolen and implies skepticism about accountability, using ""خط احمر"" (red line) to suggest untouchability or immunity. The tone is critical and skeptical.\n- Sentiment: Negative\n\nPredicted Sentiment: Negative",1
"Thoughts:\n- Positive words: None\n- Negative words: صعب, محتاج مساعده\n- Emotional Balance: The sentence expresses a sense of difficulty and a need for help, indicating a negative situation.\n- Sentiment: Negative\n\nPredicted Sentiment: Negative",1


In [ ]:
pred['Prediction'].isnull().sum()

np.int64(0)

In [ ]:
def get_text_after_sentiment(row):
    text = row['Prediction']
    word = 'Predicted Sentiment:'
    idx = text.find(word)
    if idx != -1:
        return text[idx + len(word):].strip()
    else:
      print(text)
      return 'Neutral'

# Apply to create new column
pred['Normalized Output'] = pred.apply(get_text_after_sentiment, axis=1)

pred.head()

Thoughts:
- Positive words: جعلني فداها (expresses affection or admiration)
- Negative words: محد يعطيني وجه (indicates feeling ignored or unimportant)
- Emotional Balance: The sentence expresses a mix of emotions. There is a sense of being ignored by most people on Twitter, which is negative, but there is also affection and appreciation for بسومي, who responds consistently, which is positive. The use of the crying emoji 😭 suggests a stronger emotional tone, potentially leaning towards feeling overwhelmed or emotional.

Sentiment: Negative


,Unnamed: 0,text,Label,Prediction,Normalized Output
0,0,لقد ضلم ماكرون الشعب المصرى وساهم في قمعه وقتل...,Negative,Thoughts:\n- Positive words: None\n- Negative ...,Negative
1,1,وش ذنبه يعيش هالحياة ؟ 💔 ذي نهاية الحب اللي اش...,Negative,Thoughts:\n- Positive words: None\n- Negative ...,Negative
2,2,@mhm8889 @turkialhussini1 في ناس جاتهم امراض م...,Negative,Thoughts:\n- Positive words: None\n- Negative ...,Negative
3,3,أنا بخلص رياضة من هون وبلاقي جارتنا جايبه صحن ...,Negative,Thoughts:\n- Positive words: None\n- Negative ...,Negative
4,4,بقيت احس ان المرض في مصر آخرته الموت مش العلاج...,Negative,Thoughts:\n- Positive words: None\n- Negative ...,Negative


In [ ]:
pred['Normalized Output'].value_counts()

,count
Normalized Output,
Negative,203
Positive,166
Neutral,132


In [ ]:
pred['Normalized Output'].isnull().sum()

np.int64(0)

In [ ]:
import numpy as np

preds = []
for answer in pred['Normalized Output']:
    if (
          "Positive" in answer
          or "positive" in answer
          or "pos" in answer
          or "ايجابي" in answer
          or "ايجابية" in answer
      ):
        preds.append("Positive")
    elif (
          "Negative" in answer
          or "negative" in answer
          or "neg" in answer
          or "سلبي" in answer
          or "سلبية" in answer
      ):
        preds.append("Negative")
    elif (
          "Neutral" in answer
          or "neutral" in answer
          or "neu" in answer
          or "حيادي" in answer
          or "حيادية" in answer
      ):
        preds.append("Neutral")
    else:
      preds.append('None')

In [ ]:
np.unique(preds)

array(['Negative', 'Neutral', 'Positive'], dtype='<U8')

In [ ]:
from sklearn.metrics import (f1_score,
                             precision_score,
                             recall_score,
                             classification_report,
                             confusion_matrix)

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score

labels = ['Positive', 'Negative', 'Neutral']
mapping = {'Neutral': 0, 'Negative': 1, 'Positive':2}
def map_func(x):
    return mapping.get(x, 1)

y_true = np.vectorize(map_func)(cot.Label)
y_pred = np.vectorize(map_func)(preds)

# Calculate accuracy
accuracy = accuracy_score(y_true=y_true, y_pred=y_pred)
print(f'Accuracy: {accuracy:.3f}')

# Generate accuracy report
unique_labels = set(y_true)  # Get unique labels

f1t = f1_score(y_true=y_true, y_pred=y_pred, average = 'weighted')
print('\nf1_score: ', f1t)

prec = precision_score(y_true=y_true, y_pred=y_pred, average = 'weighted')
print('\precision: ', prec)

recall = recall_score(y_true=y_true, y_pred=y_pred, average = 'weighted')
print('\recall: ', recall)

# Generate classification report
class_report = classification_report(y_true=y_true,
                                     y_pred=y_pred,
                                     digits = 4,
                                     target_names=['Neutral', 'Negative', 'Positive'])
print('\nClassification Report:')
print(class_report)

Accuracy: 0.735

f1_score:  0.7278234756495625
\precision:  0.7318554708459154
ecall:  0.7345309381237525

Classification Report:
              precision    recall  f1-score   support

     Neutral     0.6894    0.5449    0.6087       167
    Negative     0.7291    0.8862    0.8000       167
    Positive     0.7771    0.7725    0.7748       167

    accuracy                         0.7345       501
   macro avg     0.7319    0.7345    0.7278       501
weighted avg     0.7319    0.7345    0.7278       501

